# Crescendo — Spike Report (v0.2)

**Headline question:** *Is **organic** emerging-artist breakout predictable from no-audio momentum features on self-collected YouTube data — proven leakage-safe, **and** proven not to be chasing inflated (bought/bot) growth — before building any game?*

This notebook imports the `crescendo` package and renders the answer: precision@k **and the authenticity-aware headline metrics** (`organic_precision@k`, `organic_lift`, `inorganic_rate@k`) vs base-rate and naive-momentum baselines on a **temporal, per-fold** holdout. It reads the same `dataset` table the CLI builds — nothing is recomputed here that isn't in the pipeline.

> Runs meaningfully only after ~45 days of collected history (or a backfill). Until then it renders on whatever `dataset` rows exist. See `../../docs/research-2026-08-18-organic-breakout.md` for the reframe rationale.</cell id="7fb27b941602401d91542211134fc71a">

In [ ]:
from crescendo.config import load_config
from crescendo.dataset import dataset_version
from crescendo.db import Db
from crescendo.evaluate import evaluate

cfg = load_config()
db = Db(cfg.database_url)
df = db.read_dataset(version=dataset_version(cfg))
print(f'dataset rows: {len(df)}  version: {dataset_version(cfg)}')
df.head()

In [ ]:
# Walk-forward evaluation: model vs baselines, raw + ORGANIC precision@k per fold.
results = evaluate(cfg, db, cutoff=cfg.cutoff, k='auto', walk_forward=True)
for r in results:
    print(f'fold {r.fold_index} @ {r.cutoff}: P@{r.k}={r.precision_at_k:.3f} '
          f'base={r.base_rate:.3f} lift={r.lift:.2f} auc={r.roc_auc:.3f}')
    print(f'    organic: P@{r.k}={r.organic_precision_at_k:.3f} '
          f'base={r.organic_base_rate:.3f} lift={r.organic_lift:.2f} '
          f'| inorganic@{r.k}={r.inorganic_rate_at_k:.3f}')

## Verdict

- **organic_lift > 1** across folds → momentum features carry real breakout signal *and* the headline picks are authentic (positive result).
- **organic_lift ≈ 1** → no edge over base rate; an honest negative result is still a valid resume story (per L1 §8).
- **inorganic_rate@k lower than the momentum baseline's** → the model resists the pumped channels a naive growth-chaser falls for (retest: −33%, research §4).

The `reasons` (feature importances, incl. `inorganic_score`) from `model.feature_importances()` feed the transparent-AI opponent in v1.1 — which can now say *"…and this artist's growth looks real."*</cell id="8dd0d8092fe74a7c96281538738b07e2">